<a href="https://colab.research.google.com/github/teederx/Data-Science-Salary-Predictor/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Download and Load The Dataset

In [7]:
import kagglehub
import pandas as pd
import os

# Step 1: download and get local path
path = kagglehub.dataset_download("saurabhshahane/data-science-jobs-salaries")

# Step 2: see what files are in there
# print(os.listdir(path))

# Step 3: load the csv using the path
df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))

print("First 5 records:", df.head())

Using Colab cache for faster access to the 'data-science-jobs-salaries' dataset.
First 5 records:   work_year experience_level employment_type                  job_title  \
0     2021e               EN              FT    Data Science Consultant   
1      2020               SE              FT             Data Scientist   
2     2021e               EX              FT       Head of Data Science   
3     2021e               EX              FT               Head of Data   
4     2021e               EN              FT  Machine Learning Engineer   

   salary salary_currency  salary_in_usd employee_residence  remote_ratio  \
0   54000             EUR          64369                 DE            50   
1   60000             EUR          68428                 GR           100   
2   85000             USD          85000                 RU             0   
3  230000             USD         230000                 RU            50   
4  125000             USD         125000                 US       

Clean up the Data

In [8]:
new_df = df.drop(['work_year', 'job_title', 'salary', 'salary_currency', 'employee_residence', 'company_location'], axis=1)
new_df.head()

,experience_level,employment_type,salary_in_usd,remote_ratio,company_size
0,EN,FT,64369,50,L
1,SE,FT,68428,100,L
2,EX,FT,85000,0,M
3,EX,FT,230000,50,L
4,EN,FT,125000,100,S


In [9]:
from sklearn.preprocessing import StandardScaler

# Manually map ordinal columns in the correct order
# LabelEncoder assigns alphabetically which can be wrong
new_df['experience_level'] = new_df['experience_level'].map(
    {'EN': 0, 'MI': 1, 'SE': 2, 'EX': 3}  # Entry → Mid → Senior → Executive
)
new_df['company_size'] = new_df['company_size'].map(
    {'S': 0, 'M': 1, 'L': 2}  # Small → Medium → Large
)

# OneHotEncode employment_type (no natural order)
new_df = pd.get_dummies(new_df, columns=['employment_type'], drop_first=True)

new_df.head()

,experience_level,salary_in_usd,remote_ratio,company_size,employment_type_FL,employment_type_FT,employment_type_PT
0,0,64369,50,2,False,True,False
1,2,68428,100,2,False,True,False
2,3,85000,0,1,False,True,False
3,3,230000,50,2,False,True,False
4,0,125000,100,0,False,True,False


In [10]:
target_column = 'salary_in_usd'
features = new_df.drop(target_column, axis=1)
labels = new_df[target_column]  # ✅ leave salary unscaled

combined = pd.concat([features, labels], axis=1)
combined.to_csv("cleaned_dataset.csv", index=False)

We create our model using OOP

In [11]:
import torch
import numpy as np
from torch.utils.data import Dataset, random_split, DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
from torchmetrics import Accuracy
import torch.nn.init as init
from torchmetrics.regression import R2Score

Create a class ScientistDataset()

In [12]:
class ScientistDataset(Dataset):
  def __init__(self, csv_file):
    super().__init__()
    df = pd.read_csv(csv_file)  # load the CSV into a pandas dataframe
    self.data = df.to_numpy()  # convert to numpy array for faster indexing

  def __len__(self):
    # returns the total number of samples in the dataset
    return self.data.shape[0]

  def __getitem__(self, idx):
    # all columns except the last one are features
    features = torch.tensor(self.data[idx, :-1], dtype=torch.float32)
    # last column is the target salary label
    label = torch.tensor(self.data[idx, -1], dtype=torch.float32)
    # dtype=torch.float32 ensures compatibility with the PyTorch model
    return features, label

Using the instance of the class, create a dataset and split data 80% for training and 20% for testing

In [13]:
# Create the full dataset from the cleaned CSV file
dataset = ScientistDataset("cleaned_dataset.csv")

# Calculate how many samples go into training (80%) and testing (20%)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

# Randomly split the full dataset
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Extract raw numpy arrays from each split
train_features = np.array([dataset.data[i, :-1] for i in train_dataset.indices])
test_features  = np.array([dataset.data[i, :-1] for i in test_dataset.indices])
train_labels   = np.array([dataset.data[i, -1]  for i in train_dataset.indices])
test_labels    = np.array([dataset.data[i, -1]  for i in test_dataset.indices])

# Fit scaler on training data only, then apply to both
feature_scaler = StandardScaler()
label_scaler = StandardScaler()

train_features = feature_scaler.fit_transform(train_features)  # learn scale from train
test_features  = feature_scaler.transform(test_features)       # apply same scale to test
train_labels = label_scaler.fit_transform(train_labels.reshape(-1, 1)).flatten()
test_labels  = label_scaler.transform(test_labels.reshape(-1, 1)).flatten()

# Convert scaled arrays back to PyTorch TensorDatasets
train_data = TensorDataset(
    torch.tensor(train_features, dtype=torch.float32),
    torch.tensor(train_labels,   dtype=torch.float32)
)
test_data = TensorDataset(
    torch.tensor(test_features, dtype=torch.float32),
    torch.tensor(test_labels,   dtype=torch.float32)
)

# Wrap in DataLoaders
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=32, shuffle=False)

In [14]:
# Grab the first batch of 32 samples from the training loader
# 'features' contains the input data and 'label' contains the corresponding targets
features, label = next(iter(train_loader))

# print(f'Features: {features},\nLabel: {label}')
# features.shape

In [15]:
class Net(nn.Module):
  def __init__(self):
    super().__init__()
    # Fully connected layers: 4 input features → gradually narrowing → 1 salary output
    self.fc1 = nn.Linear(6, 32) #6 is the features input shape
    self.bn1 = nn.BatchNorm1d(32)
    self.fc2 = nn.Linear(32, 16)
    self.bn2 = nn.BatchNorm1d(16)
    self.fc3 = nn.Linear(16, 8)
    self.bn3 = nn.BatchNorm1d(8)
    self.fc4 = nn.Linear(8, 1)  # single output: predicted salary

    # Kaiming initialization for all layers to improve training stability
    init.kaiming_normal_(self.fc1.weight, nonlinearity='relu')
    init.kaiming_normal_(self.fc2.weight, nonlinearity='relu')
    init.kaiming_normal_(self.fc3.weight, nonlinearity='relu')
    init.kaiming_normal_(self.fc4.weight, nonlinearity='relu')

  def forward(self, x):
    # Pass through each layer: linear → normalize → activate
    x = nn.functional.relu(self.bn1(self.fc1(x)))
    x = nn.functional.relu(self.bn2(self.fc2(x)))
    x = nn.functional.relu(self.bn3(self.fc3(x)))
    # No activation on final layer — raw salary value output (regression)
    x = self.fc4(x)
    return x

In [16]:
# Initialize the network
net = Net()

criterion = nn.HuberLoss()  # handles extreme salary outliers better than MSELoss
optimizer = optim.Adam(net.parameters(), lr=0.0001)  # Adam adapts learning rate automatically

r2 = R2Score()  # R² measures prediction quality for regression (1.0 = perfect, 0.0 = poor)

for epoch in range(10000):
  for features, labels in train_loader:
    optimizer.zero_grad()  # reset gradients to avoid accumulation from previous batch
    outputs = net(features)  # forward pass — model predicts salaries
    loss = criterion(outputs, labels.view(-1, 1))  # measure how far predictions are from actual salaries
    loss.backward()  # backpropagation — compute gradients for each weight
    torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
    optimizer.step()  # update weights using computed gradients
    r2.update(outputs, labels.view(-1, 1))  # accumulate predictions for R² calculation

  if epoch % 100 == 0:  # print every 100 epochs to monitor progress
    print(f'Epoch {epoch}, Loss: {loss.item():.4f}, R2 Score: {r2.compute():.4f}')
    r2.reset()  # clear accumulated predictions before next 100 epochs

Epoch 0, Loss: 1.0274, R2 Score: -0.1880
Epoch 100, Loss: 0.5627, R2 Score: 0.0053
Epoch 200, Loss: 0.1487, R2 Score: 0.1731
Epoch 300, Loss: 0.1113, R2 Score: 0.2604
Epoch 400, Loss: 0.2275, R2 Score: 0.2764
Epoch 500, Loss: 0.1201, R2 Score: 0.2880
Epoch 600, Loss: 0.0845, R2 Score: 0.2911
Epoch 700, Loss: 1.3417, R2 Score: 0.2905
Epoch 800, Loss: 0.2117, R2 Score: 0.2950
Epoch 900, Loss: 0.0069, R2 Score: 0.3064
Epoch 1000, Loss: 0.0458, R2 Score: 0.3053
Epoch 1100, Loss: 0.0818, R2 Score: 0.3065
Epoch 1200, Loss: 0.3730, R2 Score: 0.3055
Epoch 1300, Loss: 0.2103, R2 Score: 0.3089
Epoch 1400, Loss: 0.2268, R2 Score: 0.3084
Epoch 1500, Loss: 0.9507, R2 Score: 0.3108
Epoch 1600, Loss: 0.1007, R2 Score: 0.3101
Epoch 1700, Loss: 0.2398, R2 Score: 0.3151
Epoch 1800, Loss: 0.5091, R2 Score: 0.3160
Epoch 1900, Loss: 0.1756, R2 Score: 0.3189
Epoch 2000, Loss: 0.1819, R2 Score: 0.3172
Epoch 2100, Loss: 0.2660, R2 Score: 0.3180
Epoch 2200, Loss: 1.6682, R2 Score: 0.3207
Epoch 2300, Loss: 0.09

In [17]:
r2 = R2Score()

net.eval()

with torch.no_grad():
  for features, labels in test_loader:
    outputs = net(features)
    r2.update(outputs, labels.view(-1, 1))

print(f'Test Accuracy: {r2.compute():.4f}')

Test Accuracy: 0.1618


In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Train the model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(train_features, train_labels)

# Evaluate on training data
train_preds = rf_model.predict(train_features)
train_r2 = r2_score(train_labels, train_preds)

# Evaluate on test data
test_preds = rf_model.predict(test_features)
test_r2 = r2_score(test_labels, test_preds)

print(f'Train R2 Score: {train_r2:.4f}')
print(f'Test R2 Score:  {test_r2:.4f}')

Train R2 Score: 0.3864
Test R2 Score:  0.0254


In [22]:
# Configure git with your details
os.system('git config --global user.email "idowufavour07@gmail.com"')
os.system('git config --global user.name "teederx"')

# Initialize and push
os.system('git init')
os.system('git add .')
os.system('git commit -m "Initial commit"')
os.system('git branch -M main')
os.system('git remote add origin https://github.com/teederx/Data-Science-Salary-Predictor.git')
os.system('git push -u origin main')

32768